In [9]:
!pip install langchain_community
!pip install langchain_experimental
!pip install python-dotenv
!pip install langchain_mistralai
!pip install pypdf
!pip install pinecone
!pip install langchain_pinecone
!pip install redis
!pip install redisvl
!pip install langchain-redis
!pip install sentence_transformers

In [2]:

from dotenv import load_dotenv
load_dotenv()

import os
from getpass import getpass

# os.environ["PINECONE_API_KEY"] = getpass("Enter your Pinecone API key: ")
# os.environ["MISTRAL_API_KEY"] = getpass("Enter your Mistral API key: ")

from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader('/content/llm.pdf')
pages=loader.load()
# print(pages[0].page_content)

print(len(pages))
print(pages[1].metadata)
key_metadata=['description-abstract','producer','eventtype','created','creator','date','creationdate','firstpage','lastpage','subject','language','moddate','description','editors']


for doc in pages:
    for key in key_metadata:
        doc.metadata.pop(key,None)

11
{'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-Frenc

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_mistralai import MistralAIEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain_mistralai import MistralAIEmbeddings
import os

MISTRAL_API_KEY="MISTRAL_API_KEY"
embeddings = MistralAIEmbeddings(
    model="mistral-embed",
    api_key=MISTRAL_API_KEY
)

splitterS = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=80
)

semantic_chunks = splitterS.split_documents(pages)
print(len(semantic_chunks[0].page_content))
print(semantic_chunks[0].metadata)



llm = ChatMistralAI(
    model="mistral-medium-3-5",
    api_key=MISTRAL_API_KEY
)


embeddings = MistralAIEmbeddings(
    model="mistral-embed",
    api_key=MISTRAL_API_KEY
)
embedding_vectors = embeddings.embed_documents(
    [doc.page_content for doc in semantic_chunks]
)

print(len(embedding_vectors))      # Number of chunks
print(len(embedding_vectors[0]))   # Embedding dimension
print(embedding_vectors[0])




In [51]:

from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model="mistral-large-latest",
    api_key=MISTRAL_API_KEY
)

In [9]:
from google.colab import userdata

REDIS_HOST = "redis-16333.c264.ap-south-1-1.ec2.cloud.redislabs.com"
REDIS_PORT = 16233
REDIS_PASSWORD = userdata.get('REDIS_PASSWORD')  # set this in Colab's 🔑 Secrets panel

In [15]:
import redis, time

r = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password="password",
    decode_responses=True
)

t0 = time.perf_counter()
print(r.ping())
t1 = time.perf_counter()
print(f"Round trip: {(t1-t0)*1000:.1f} ms")

True
Round trip: 1183.5 ms


In [18]:


redis_url = f"redis://default:{REDIS_PASSWORD}@{REDIS_HOST}:{REDIS_PORT}"

from langchain_redis import RedisVectorStore

vectorstore = RedisVectorStore.from_documents(
    documents=semantic_chunks,
    embedding=embeddings,
    redis_url=redis_url,
    index_name="llm_paper",
    algorithm="HNSW",
    distance_metric="COSINE",
    # storage_type="json"
)

In [21]:
from langchain_redis import RedisVectorStore

vectorstore = RedisVectorStore(
    embeddings=embeddings,
    redis_url=redis_url,
    index_name="llm_paper"
)

In [32]:
from sentence_transformers import CrossEncoder

reranker1= CrossEncoder("BAAI/bge-reranker-base")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [53]:
import time

# ------------------ Total Pipeline ------------------
pipeline_start = time.perf_counter()

question = "what is Encoder and Decoder Stacks"


# ------------------ Query Transformation ------------------
qt_start = time.perf_counter()

rewrite_prompt = f"""Rewrite this as a concise search query. Output only the query, nothing else.

Query: {question}"""

# rewritten_query = llm.invoke(rewrite_prompt).content.strip()
def needs_rewrite(q: str) -> bool:
    return len(q.split()) > 12

if needs_rewrite(question):
    rewritten_query =llm.invoke(rewrite_prompt).content.strip()
else:
    rewritten_query = question
qt_end = time.perf_counter()

# ------------------ Redis Retrieval ------------------
retrieval_start = time.perf_counter()

results3 = vectorstore.similarity_search(
    query=rewritten_query,
    k=5
)

retrieval_end = time.perf_counter()

# ------------------ Prepare Pairs ------------------
pair_start = time.perf_counter()

pairs = [
    [rewritten_query, doc.page_content]
    for doc in results3
]

pair_end = time.perf_counter()

# ------------------ Cross Encoder ------------------
rerank_start = time.perf_counter()

scores = reranker1.predict(pairs)

ranked = sorted(
    zip(results3, scores),
    key=lambda x: x[1],
    reverse=True
)

results = [doc for doc, score in ranked[:5]]

rerank_end = time.perf_counter()

# ------------------ Context Creation ------------------
context_start = time.perf_counter()

context = "\n\n".join(
    f"""
Source: {doc.metadata.get('source')}
Page: {doc.metadata.get('page')}

{doc.page_content}
"""
    for doc in results
)

prompt = f"""
Answer only from the context below.
If the answer is not present, say "I don't have the answer."

Context:
{context}

Question:
{question}
"""

context_end = time.perf_counter()

# ------------------ LLM ------------------
llm_start = time.perf_counter()

answer = llm.invoke(prompt)

llm_end = time.perf_counter()

# ------------------ Total ------------------
pipeline_end = time.perf_counter()

print("="*50)
print(f"Query Transformation : {qt_end - qt_start:.3f} sec")
print(f"Redis Retrieval      : {retrieval_end - retrieval_start:.3f} sec")
print(f"Pair Creation        : {pair_end - pair_start:.3f} sec")
print(f"Cross Encoder        : {rerank_end - rerank_start:.3f} sec")
print(f"Context Creation     : {context_end - context_start:.3f} sec")
print(f"LLM Generation       : {llm_end - llm_start:.3f} sec")
print("-"*50)
print(f"Total Pipeline       : {pipeline_end - pipeline_start:.3f} sec")
print("="*50)

print("\nAnswer:\n")
print(answer.content)

Query Transformation : 0.000 sec
Redis Retrieval      : 1.145 sec
Pair Creation        : 0.000 sec
Cross Encoder        : 0.150 sec
Context Creation     : 0.000 sec
LLM Generation       : 3.320 sec
--------------------------------------------------
Total Pipeline       : 4.616 sec

Answer:

From the provided context:

**Encoder and Decoder Stacks** refer to the layered structures in the Transformer model:

- **Encoder Stack**: Composed of **N = 6 identical layers**. Each layer has two sub-layers: a self-attention mechanism and a position-wise fully connected feed-forward network. Residual connections and layer normalization are applied around each sub-layer.

- **Decoder Stack**: Also composed of **N = 6 identical layers**, but each layer has **three sub-layers**: the two found in the encoder (self-attention and feed-forward) plus an additional **multi-head attention sub-layer** that attends to the encoder’s output. Like the encoder, residual connections and layer normalization are use

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 65.8 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.2.0
    Uninstalling hf-xet-1.2.0:
      Successfully uninstalled hf-xet-1.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-redis 0.2.5 requires hf-xet<1.2.1, but you have hf-xet 1.5.1 which is incompatible.
